# Gauss-Legendre check (Rust)

Start JupyterLab with `workshop run -- notebook`. Uses the `thole` facade crate from the working tree. After changing the Rust code, run the `:dep` cell again.

In [ ]:
:dep thole = { path = "/project/crates/thole" }

In [ ]:
use thole::quad::{gauss_legendre, Error, GaussLegendre};

## 1. The rule with 3 points

The Python binding runs this same code, so the bits must equal the `hex` printed by the Python notebook.

In [ ]:
let rule = GaussLegendre::new(3).unwrap();
println!("{:?} {:?}", rule.nodes(), rule.weights());
println!("bits = {:#018x} {:#018x}", rule.nodes()[2].to_bits(), rule.weights()[2].to_bits());
println!("node error = {:e}", rule.nodes()[2] - 0.6_f64.sqrt());

## 2. What the Lean theorems predict

`gauss_exact`, `rule_exact_of_exact`: exact for every polynomial of degree at most `2n - 1`, and not for `2n`.

In [ ]:
let (a, b): (f64, f64) = (-0.5, 2.0);
for n in 1..=10 {
    let rule = GaussLegendre::new(n).unwrap();
    let errors: Vec<f64> = [2 * n - 1, 2 * n].iter().map(|&d| {
        let exact = (b.powi(d as i32 + 1) - a.powi(d as i32 + 1)) / (d as f64 + 1.0);
        ((rule.integrate(|x| x.powi(d as i32), a, b).unwrap() - exact) / exact).abs()
    }).collect();
    println!("n={n:2}  degree 2n-1: {:.2e}  degree 2n: {:.2e}", errors[0], errors[1]);
}

## 3. Reference integrals (mpmath, 50 digits)

In [ ]:
let cases: [(&str, fn(f64) -> f64, f64, f64, f64); 3] = [
    ("exp", f64::exp, 0.0, 1.0, 1.7182818284590453),
    ("cos 30x", |x| (30.0 * x).cos(), 0.0, 1.0, -0.03293438746976206),
    ("exp(-x^2)", |x| (-x * x).exp(), -5.0, 5.0, 1.772453850902791),
];
for (name, f, a, b, expected) in cases {
    for n in [10, 20, 40] {
        let v = gauss_legendre(f, a, b, n).unwrap();
        println!("{name:10} n={n:2} value={v:.17} error={:.1e}", (v - expected).abs());
    }
}

## 4. Failures

In [ ]:
let results: Vec<(&str, Result<f64, Error>)> = vec![
    ("zero points", gauss_legendre(f64::exp, 0.0, 1.0, 0)),
    ("infinite end", gauss_legendre(f64::exp, 0.0, f64::INFINITY, 5)),
    ("f returns inf", gauss_legendre(|x| 1.0 / x, -1.0, 1.0, 5)),
];
for (name, r) in &results {
    println!("{name:14} -> {r:?}");
}